<a href="https://colab.research.google.com/github/TrishaThanmai/sample1/blob/main/Initial_Data_Processing_and_IPQ_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 1. Install required lib (if not already)
!pip install -q ipynbname

# 2. Define filename for the zip
zip_name = "Initial Data Processing and IPQ Extraction.zip"

# 3. Create a zip of your notebook file
# The notebook name can be fetched dynamically using the Colab metadata trick
import os
import ipynbname

try:
    # Get the notebook name dynamically
    nb_name = ipynbname.name() + '.ipynb'
    nb_path = os.path.join('/content/', nb_name)

    # Check if the notebook exists before zipping
    if os.path.exists(nb_path):
        # Create a zip of your notebook file
        os.system(f'zip -r /content/{zip_name} {nb_path}')

        # 4. Download the zip
        from google.colab import files
        files.download(f'/content/{zip_name}')
    else:
        print(f"Error: Notebook file not found at {nb_path}. Please ensure the notebook is saved.")

except FileNotFoundError:
    print("Error: ipynbname could not detect the notebook name. Please ensure you are running this in a Colab notebook environment.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.2 MB/s eta 0:00:00
Error: Notebook file not found at /content/fileId=1RA9nYNGfW7OCilnWASONbqTVlBINQv7G.ipynb. Please ensure the notebook is saved.


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder

# --- 1. CORE FEATURE EXTRACTION FUNCTION ---

def extract_ipq_and_unit(df):
    """
    Extracts structured features (Value and Unit) from the end of the 'catalog_content' string,
    which represents the Item Pack Quantity (IPQ) and its unit of measure.
    """

    # Regex pattern to find the 'Value' and 'Unit' fields, typically at the end of the content.
    # It looks for "Value: [number]" and "Unit: [text]" separated by a newline.
    # We use a non-greedy match (.*?) to handle potential variability.
    pattern = re.compile(r'Value:\s*([\d\.]+)\s*\nUnit:\s*(.*?)(?:\n|$)', re.DOTALL | re.IGNORECASE)

    # Initialize new columns
    df['item_pack_quantity'] = np.nan
    df['quantity_unit'] = 'unknown'
    df['catalog_content_cleaned'] = df['catalog_content'].copy()

    for index, row in df.iterrows():
        content = str(row['catalog_content']).strip()

        match = pattern.search(content)

        if match:
            # Group 1 is the numerical value
            try:
                df.loc[index, 'item_pack_quantity'] = float(match.group(1))
            except ValueError:
                # Handle cases where value might be non-numeric if regex fails
                df.loc[index, 'item_pack_quantity'] = np.nan

            # Group 2 is the unit string
            unit_raw = match.group(2).strip()
            # Clean up the unit (e.g., remove trailing spaces or stray characters)
            df.loc[index, 'quantity_unit'] = unit_raw.lower()

            # Remove the extracted fields from the content for cleaner NLP later
            df.loc[index, 'catalog_content_cleaned'] = content[:match.start()]

        else:
            # If no match, IPQ defaults to 1 or NaN, and unit is 'unknown'
            df.loc[index, 'item_pack_quantity'] = 1.0

    return df

# --- 2. DATA LOADING AND PROCESSING EXECUTION ---

# Load the available sample data (Simulating 'test.csv' for feature development)
try:
    # Assuming 'sample_test.csv' has the same structure as the final 'test.csv'
    df_data = pd.read_csv('/content/sample_test.csv')

    # Apply the IPQ/Unit extraction
    df_processed = extract_ipq_and_unit(df_data)

    # --- 3. ENCODING CATEGORICAL UNIT FEATURE ---

    # Since quantity_unit is a critical feature, we encode it for the ML model.
    # We'll use Label Encoding for demonstration, suitable for tree-based models like LightGBM.
    le = LabelEncoder()
    # Fill NaN values in quantity_unit before encoding, though 'unknown' is already a category
    df_processed['quantity_unit_encoded'] = le.fit_transform(df_processed['quantity_unit'].astype(str))

    # --- 4. DISPLAY RESULTS ---

    print("--- Extracted and Engineered Features (Sample Data) ---")
    print("Original 'catalog_content' is now split into 3 useful features:")
    print("1. item_pack_quantity (Numerical IPQ)")
    print("2. quantity_unit (Categorical Unit of Measure)")
    print("3. catalog_content_cleaned (Remaining text for NLP)\n")

    # Display key columns to verify extraction
    print(df_processed[['sample_id', 'item_pack_quantity', 'quantity_unit', 'quantity_unit_encoded']].head(10))

    print(f"\nUnique Units Extracted: {df_processed['quantity_unit'].nunique()} ({df_processed['quantity_unit'].unique()[:5]}...)")

except FileNotFoundError:
    print("Error: Required file 'sample_test.csv' not found. Please ensure the training and test data files are available.")
except Exception as e:
    print(f"An error occurred during processing: {e}")

--- Extracted and Engineered Features (Sample Data) ---
Original 'catalog_content' is now split into 3 useful features:
1. item_pack_quantity (Numerical IPQ)
2. quantity_unit (Categorical Unit of Measure)
3. catalog_content_cleaned (Remaining text for NLP)

   sample_id  item_pack_quantity quantity_unit  quantity_unit_encoded
0     217392                1.00         count                      0
1     209156               16.00         ounce                      2
2     262333               72.00         fl oz                      1
3     295979               10.10         fl oz                      1
4      50604               10.00         count                      0
5     184343               16.00         count                      0
6      11241                0.50         ounce                      2
7      17551               11.15         fl oz                      1
8      13187               40.00         ounce                      2
9      55004               12.00         o

In [ ]:
import pandas as pd
import numpy as np
import hashlib

# Define the expected dimension for the image features (e.g., ResNet-50 output)
IMAGE_FEATURE_DIM = 512
IMAGE_FEATURE_COLS = [f'img_feat_{i}' for i in range(IMAGE_FEATURE_DIM)]

def generate_image_features(df, base_path='images/'):
    """
    Mocks the process of downloading images and extracting visual features using a CNN.

    In a real scenario, this function would:
    1. Call src/utils.py's download_images(df['image_link']) to save files locally.
    2. Load a pre-trained CNN (e.g., ResNet-50).
    3. Process each image through the CNN to get the bottleneck layer output (e.g., 512 or 2048 dimensions).

    For the mock, we generate deterministic, random-like features based on the image_link hash.
    """
    print(f"--- Mocking Image Feature Extraction ({IMAGE_FEATURE_DIM} dimensions) ---")

    # 1. Create a stable seed for feature generation based on the image_link
    df['link_hash'] = df['image_link'].apply(lambda x: int(hashlib.sha256(str(x).encode('utf-8')).hexdigest(), 16) % (2**32))

    # 2. Generate feature columns
    feature_matrix = np.zeros((len(df), IMAGE_FEATURE_DIM))

    for i in range(len(df)):
        # Use the hash as a local seed to make the features reproducible for the same link
        seed = df.loc[i, 'link_hash']
        np.random.seed(seed)

        # Generate random features, normalized to roughly (0, 1)
        feature_matrix[i, :] = np.random.rand(IMAGE_FEATURE_DIM)

    # 3. Add the generated features back to the DataFrame
    df[IMAGE_FEATURE_COLS] = feature_matrix

    # Drop the temporary hash column
    df = df.drop(columns=['link_hash'])

    print(f"Successfully mocked and added {IMAGE_FEATURE_DIM} visual features.")
    return df

if __name__ == '__main__':
    # Load the available sample data
    try:
        df_data = pd.read_csv('sample_test.csv')
        df_featured = generate_image_features(df_data)

        print("\n--- Verification of Feature Shape ---")
        print(f"DataFrame shape after adding image features: {df_featured.shape}")
        print(f"First 5 feature values for the first sample:")
        print(df_featured[IMAGE_FEATURE_COLS].iloc[0].head(5))

    except FileNotFoundError:
        print("Error: Required file 'sample_test.csv' not found.")


Processing train.csv with 100 samples...
-> Extracting Structured Features (IPQ/Unit)...
-> Mocking Text Embedding Generation (768D)...


/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

-> Mocking Image Feature Generation (512D)...


/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[IMAGE_FEATURE_COLS] = feature_matrix
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[IMAGE_FEATURE_COLS] = feature_matrix
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use 


Processing test.csv with 100 samples...
-> Extracting Structured Features (IPQ/Unit)...
-> Mocking Text Embedding Generation (768D)...


/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

-> Mocking Image Feature Generation (512D)...


/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[IMAGE_FEATURE_COLS] = feature_matrix
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[IMAGE_FEATURE_COLS] = feature_matrix
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use 


--- Starting 5-Fold Multimodal LightGBM Training ---
Fold 1 Validation RMSE (Actual Price): $37.14
Fold 2 Validation RMSE (Actual Price): $22.43
Fold 3 Validation RMSE (Actual Price): $33.46
Fold 4 Validation RMSE (Actual Price): $17.88
Fold 5 Validation RMSE (Actual Price): $35.40

Average Actual Price RMSE across 5 folds: $29.26

--- FINAL SUBMISSION GENERATED ---
File: test_out.csv (Ready for upload)
First 5 predictions:
   sample_id      price
0     217392  20.290081
1     209156  17.267707
2     262333  21.279687
3     295979  24.530902
4      50604  15.932095


In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib

# Define the expected dimension for the Text Features (e.g., BERT embedding size)
TEXT_EMBEDDING_DIM = 768
TEXT_EMBEDDING_COLS = [f'text_feat_{i}' for i in range(TEXT_EMBEDDING_DIM)]

# --- 1. UTILITY: IPQ Extraction (Needed to clean the content first) ---

def extract_ipq_and_unit(df):
    """
    Utility function to clean the content by removing the IPQ/Unit fields,
    leaving only the descriptive text for the embedding model.
    (Copied from previous step for modularity)
    """
    pattern = re.compile(r'Value:\s*([\d\.]+)\s*\nUnit:\s*(.*?)(?:\n|$)', re.DOTALL | re.IGNORECASE)
    df['catalog_content_cleaned'] = df['catalog_content'].copy()

    for index, row in df.iterrows():
        content = str(row['catalog_content']).strip()
        match = pattern.search(content)

        if match:
            # Remove the matched IPQ/Unit section to isolate descriptive text
            df.loc[index, 'catalog_content_cleaned'] = content[:match.start()].strip()
        else:
            # Ensure any non-string content is handled
            df.loc[index, 'catalog_content_cleaned'] = content.strip()

    return df

# --- 2. CORE: MOCK EMBEDDING GENERATION FUNCTION ---

def generate_text_embeddings(df):
    """
    Mocks the process of using a pre-trained language model (e.g., BERT, RoBERTa)
    to generate dense semantic embeddings for the product description.

    In a real scenario, this function would:
    1. Tokenize the df['catalog_content_cleaned'].
    2. Pass tokens through a BERT model to get the [CLS] token output vector.
    """
    print(f"--- Mocking Text Embedding Generation ({TEXT_EMBEDDING_DIM} dimensions) ---")

    # 1. Ensure text is handled (fill NaN/empty strings)
    df['catalog_content_cleaned'] = df['catalog_content_cleaned'].fillna('')

    # 2. Create a stable seed for feature generation based on the text hash
    df['text_hash'] = df['catalog_content_cleaned'].apply(
        lambda x: int(hashlib.sha256(x.encode('utf-8')).hexdigest(), 16) % (2**32)
    )

    # 3. Generate feature columns
    embedding_matrix = np.zeros((len(df), TEXT_EMBEDDING_DIM))

    for i in range(len(df)):
        # Use the hash as a local seed to make the embeddings reproducible
        seed = df.loc[i, 'text_hash']
        np.random.seed(seed)

        # Generate random features normalized to roughly (-1, 1), typical of embeddings
        embedding_matrix[i, :] = np.random.uniform(-1, 1, TEXT_EMBEDDING_DIM)

    # 4. Add the generated features back to the DataFrame
    df[TEXT_EMBEDDING_COLS] = embedding_matrix

    # Drop the temporary hash column
    df = df.drop(columns=['text_hash'])

    print(f"Successfully mocked and added {TEXT_EMBEDDING_DIM} text features.")
    return df

# --- 3. EXECUTION ---

if __name__ == '__main__':
    # Load the available sample data
    try:
        df_data = pd.read_csv('sample_test.csv')

        # Step 1: Clean the content
        df_cleaned = extract_ipq_and_unit(df_data)

        # Step 2: Generate the embeddings
        df_embedded = generate_text_embeddings(df_cleaned)

        print("\n--- Verification of Feature Shape ---")
        print(f"DataFrame shape after adding text embeddings: {df_embedded.shape}")
        print(f"First 5 embedding values for the first sample:")
        print(df_embedded[TEXT_EMBEDDING_COLS].iloc[0].head(5))

        print("\n--- Cleaned Content Sample ---")
        print(df_embedded[['sample_id', 'catalog_content_cleaned']].head(1))

    except FileNotFoundError:
        print("Error: Required file 'sample_test.csv' not found.")

--- Mocking Text Embedding Generation (768 dimensions) ---


/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

Successfully mocked and added 768 text features.

--- Verification of Feature Shape ---
DataFrame shape after adding text embeddings: (100, 772)
First 5 embedding values for the first sample:
text_feat_0   -0.408232
text_feat_1    0.357787
text_feat_2   -0.089079
text_feat_3   -0.393321
text_feat_4    0.200502
Name: 0, dtype: float64

--- Cleaned Content Sample ---
   sample_id                            catalog_content_cleaned
0     217392  Item Name: Gift Basket Village Gourmet Meat an...


/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib

# Define the expected dimension for the Text Features (e.g., BERT embedding size)
TEXT_EMBEDDING_DIM = 768
TEXT_EMBEDDING_COLS = [f'text_feat_{i}' for i in range(TEXT_EMBEDDING_DIM)]

# --- 1. UTILITY: IPQ Extraction (Needed to clean the content first) ---

def extract_ipq_and_unit(df):
    """
    Utility function to clean the content by removing the IPQ/Unit fields,
    leaving only the descriptive text for the embedding model.
    (Copied from previous step for modularity)
    """
    pattern = re.compile(r'Value:\s*([\d\.]+)\s*\nUnit:\s*(.*?)(?:\n|$)', re.DOTALL | re.IGNORECASE)
    df['catalog_content_cleaned'] = df['catalog_content'].copy()

    for index, row in df.iterrows():
        content = str(row['catalog_content']).strip()
        match = pattern.search(content)

        if match:
            # Remove the matched IPQ/Unit section to isolate descriptive text
            df.loc[index, 'catalog_content_cleaned'] = content[:match.start()].strip()
        else:
            # Ensure any non-string content is handled
            df.loc[index, 'catalog_content_cleaned'] = content.strip()

    return df

# --- 2. CORE: MOCK EMBEDDING GENERATION FUNCTION ---

def generate_text_embeddings(df):
    """
    Mocks the process of using a pre-trained language model (e.g., BERT, RoBERTa)
    to generate dense semantic embeddings for the product description.

    In a real scenario, this function would:
    1. Tokenize the df['catalog_content_cleaned'].
    2. Pass tokens through a BERT model to get the [CLS] token output vector.
    """
    print(f"--- Mocking Text Embedding Generation ({TEXT_EMBEDDING_DIM} dimensions) ---")

    # 1. Ensure text is handled (fill NaN/empty strings)
    df['catalog_content_cleaned'] = df['catalog_content_cleaned'].fillna('')

    # 2. Create a stable seed for feature generation based on the text hash
    df['text_hash'] = df['catalog_content_cleaned'].apply(
        lambda x: int(hashlib.sha256(x.encode('utf-8')).hexdigest(), 16) % (2**32)
    )

    # 3. Generate feature columns
    embedding_matrix = np.zeros((len(df), TEXT_EMBEDDING_DIM))

    for i in range(len(df)):
        # Use the hash as a local seed to make the embeddings reproducible
        seed = df.loc[i, 'text_hash']
        np.random.seed(seed)

        # Generate random features normalized to roughly (-1, 1), typical of embeddings
        embedding_matrix[i, :] = np.random.uniform(-1, 1, TEXT_EMBEDDING_DIM)

    # 4. Add the generated features back to the DataFrame
    df[TEXT_EMBEDDING_COLS] = embedding_matrix

    # Drop the temporary hash column
    df = df.drop(columns=['text_hash'])

    print(f"Successfully mocked and added {TEXT_EMBEDDING_DIM} text features.")
    return df

# --- 3. EXECUTION ---

if __name__ == '__main__':
    # Load the available sample data
    try:
        df_data = pd.read_csv('sample_test.csv')

        # Step 1: Clean the content
        df_cleaned = extract_ipq_and_unit(df_data)

        # Step 2: Generate the embeddings
        df_embedded = generate_text_embeddings(df_cleaned)

        print("\n--- Verification of Feature Shape ---")
        print(f"DataFrame shape after adding text embeddings: {df_embedded.shape}")
        print(f"First 5 embedding values for the first sample:")
        print(df_embedded[TEXT_EMBEDDING_COLS].iloc[0].head(5))

        print("\n--- Cleaned Content Sample ---")
        print(df_embedded[['sample_id', 'catalog_content_cleaned']].head(1))

    except FileNotFoundError:
        print("Error: Required file 'sample_test.csv' not found.")

--- Mocking Text Embedding Generation (768 dimensions) ---


/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

Successfully mocked and added 768 text features.

--- Verification of Feature Shape ---
DataFrame shape after adding text embeddings: (100, 772)
First 5 embedding values for the first sample:
text_feat_0   -0.408232
text_feat_1    0.357787
text_feat_2   -0.089079
text_feat_3   -0.393321
text_feat_4    0.200502
Name: 0, dtype: float64

--- Cleaned Content Sample ---
   sample_id                            catalog_content_cleaned
0     217392  Item Name: Gift Basket Village Gourmet Meat an...


/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2508739905.py:67: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

In [ ]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import hashlib

# --- MOCK FEATURE DIMENSIONS (Reflecting real BERT/ResNet outputs) ---
TEXT_EMBEDDING_DIM = 768
IMAGE_FEATURE_DIM = 512

TEXT_EMBEDDING_COLS = [f'text_feat_{i}' for i in range(TEXT_EMBEDDING_DIM)]
IMAGE_FEATURE_COLS = [f'img_feat_{i}' for i in range(IMAGE_FEATURE_DIM)]

# --- 1. FEATURE EXTRACTION UTILITIES (Mocking Modules) ---

def extract_structured_and_clean_text(df):
    """
    Extracts IPQ/Unit and cleans the catalog_content for text embedding.
    (Logic adapted from text_embedding_generator.py)
    """
    print("-> Extracting Structured Features (IPQ/Unit)...")
    pattern = re.compile(r'Value:\s*([\d\.]+)\s*\nUnit:\s*(.*?)(?:\n|$)', re.DOTALL | re.IGNORECASE)

    df['item_pack_quantity'] = 1.0
    df['quantity_unit'] = 'unknown'
    df['catalog_content_cleaned'] = df['catalog_content'].astype(str).copy()

    for index, row in df.iterrows():
        content = str(row['catalog_content']).strip()
        match = pattern.search(content)

        if match:
            try:
                # Extract numerical IPQ
                df.loc[index, 'item_pack_quantity'] = float(match.group(1))
            except ValueError:
                df.loc[index, 'item_pack_quantity'] = 1.0

            # Extract categorical Unit
            df.loc[index, 'quantity_unit'] = match.group(2).strip().lower()

            # Clean content for embedding model
            df.loc[index, 'catalog_content_cleaned'] = content[:match.start()].strip()

    # Fill any remaining NaNs with the median
    df['item_pack_quantity'] = df['item_pack_quantity'].fillna(df['item_pack_quantity'].median())

    return df

def generate_text_embeddings(df):
    """Mocks the generation of high-dimensional BERT/RoBERTa embeddings."""
    print(f"-> Mocking Text Embedding Generation ({TEXT_EMBEDDING_DIM}D)...")

    df['catalog_content_cleaned'] = df['catalog_content_cleaned'].fillna('')
    df['text_hash'] = df['catalog_content_cleaned'].apply(
        lambda x: int(hashlib.sha256(x.encode('utf-8')).hexdigest(), 16) % (2**32)
    )

    embedding_matrix = np.zeros((len(df), TEXT_EMBEDDING_DIM))

    for i in range(len(df)):
        seed = df.loc[i, 'text_hash']
        np.random.seed(seed)
        embedding_matrix[i, :] = np.random.uniform(-1, 1, TEXT_EMBEDDING_DIM)

    df[TEXT_EMBEDDING_COLS] = embedding_matrix
    return df.drop(columns=['text_hash'])

def generate_image_features(df):
    """Mocks the generation of high-dimensional ResNet/EfficientNet features."""
    print(f"-> Mocking Image Feature Generation ({IMAGE_FEATURE_DIM}D)...")

    # Use image_link hash to create stable, but random-like, visual features
    df['link_hash'] = df['image_link'].apply(lambda x: int(hashlib.sha256(str(x).encode('utf-8')).hexdigest(), 16) % (2**32))

    feature_matrix = np.zeros((len(df), IMAGE_FEATURE_DIM))

    for i in range(len(df)):
        seed = df.loc[i, 'link_hash']
        np.random.seed(seed + 1) # Use a different seed offset for image vs text
        feature_matrix[i, :] = np.random.rand(IMAGE_FEATURE_DIM)

    df[IMAGE_FEATURE_COLS] = feature_matrix
    return df.drop(columns=['link_hash'])

# --- 2. DATA LOADING AND FULL PREPROCESSING ---

def load_and_preprocess_data(file_name):
    """Loads CSV and applies all multimodal feature engineering steps."""

    # --- MOCK DATA LOADING ---
    if file_name == 'train.csv':
        # Mocking real train data (75k samples)
        df = pd.read_csv('sample_test.csv').copy()
        # Create mock target variable 'price', ensuring it's strictly positive
        np.random.seed(42)
        # Generate positive mock prices
        df['price'] = np.expm1(np.random.normal(loc=np.log(20), scale=np.log(3), size=len(df)))
        # Added safety clip to ensure all mock prices are positive before log transformation later
        df['price'] = df['price'].clip(lower=0.01)
    elif file_name == 'test.csv':
        # Mocking real test data (75k samples)
        df = pd.read_csv('sample_test.csv').copy()
    else:
        raise FileNotFoundError(f"Mock file configuration for {file_name} not found.")

    print(f"\nProcessing {file_name} with {len(df)} samples...")

    # 2.1. Structured and Text Cleaning
    df = extract_structured_and_clean_text(df)

    # 2.2. Textual Features
    df = generate_text_embeddings(df)

    # 2.3. Visual Features
    df = generate_image_features(df)

    # 2.4. Additional Tabular Feature
    df['cleaned_content_length'] = df['catalog_content_cleaned'].apply(len)

    # 2.5. Categorical Encoding (Unit)
    le = LabelEncoder()
    # In a real pipeline, the LE would be fit on the combined train/test set units to avoid data leakage
    df['quantity_unit_encoded'] = le.fit_transform(df['quantity_unit'].astype(str))
    df['quantity_unit_encoded'] = df['quantity_unit_encoded'].astype('category')

    return df

# --- 3. MODEL TRAINING AND PREDICTION ---

def train_and_predict():

    # Load and preprocess mock data
    df_train = load_and_preprocess_data('train.csv')
    df_test = load_and_preprocess_data('test.csv')

    # Target Transformation (Crucial for SMAPE/Relative Error Minimization)
    # Using Pandas Series.clip(lower=...) to explicitly clip before log1p
    df_train['log_price'] = np.log1p(df_train['price'].clip(lower=1e-6))

    # Define feature lists
    TABULAR_FEATURES = ['item_pack_quantity', 'cleaned_content_length']
    CATEGORICAL_FEATURES = ['quantity_unit_encoded']

    # Combine all feature sets for the multimodal model
    ALL_FEATURES = TABULAR_FEATURES + CATEGORICAL_FEATURES + TEXT_EMBEDDING_COLS + IMAGE_FEATURE_COLS

    X_train = df_train[ALL_FEATURES]
    y_train = df_train['log_price']
    X_test = df_test[ALL_FEATURES]

    # LightGBM Parameters (Optimized for regression with K-Fold)
    params = {
        'objective': 'regression_l1', # MAE objective helps minimize relative errors better than L2
        'metric': 'rmse',
        'n_estimators': 1500,
        'learning_rate': 0.03,
        'feature_fraction': 0.7,
        'bagging_fraction': 0.7,
        'bagging_freq': 1,
        'lambda_l1': 0.1,
        'verbose': -1,
        'n_jobs': -1,
        'seed': 42
    }

    # K-Fold Cross-Validation for Ensemble Robustness
    N_SPLITS = 5
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    test_preds_log = np.zeros(len(df_test))
    fold_rmses = []

    print(f"\n--- Starting {N_SPLITS}-Fold Multimodal LightGBM Training ---")

    for fold, (train_index, val_index) in enumerate(kf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)],
            categorical_feature=CATEGORICAL_FEATURES
        )

        # Ensemble prediction for the test set
        test_preds_log += model.predict(X_test) / N_SPLITS

        # Validation RMSE (on actual price scale)
        val_preds_log = model.predict(X_val)
        val_preds_price = np.expm1(val_preds_log)
        y_val_price = np.expm1(y_val)
        fold_rmse = np.sqrt(mean_squared_error(y_val_price, val_preds_price))
        fold_rmses.append(fold_rmse)
        print(f"Fold {fold+1} Validation RMSE (Actual Price): ${fold_rmse:,.2f}")

    avg_rmse = np.mean(fold_rmses)
    print(f"\nAverage Actual Price RMSE across {N_SPLITS} folds: ${avg_rmse:,.2f}")

    # --- 4. FINAL SUBMISSION GENERATION ---

    # Inverse transform and clip predictions
    # FIXED: Use explicit np.clip with a_min=0.01 to ensure robustness and avoid the ValueError.
    predicted_prices = np.clip(np.expm1(test_preds_log), a_min=0.01, a_max=None)

    submission_df = pd.DataFrame({
        'sample_id': df_test['sample_id'],
        'price': predicted_prices
    })

    # Ensure price column is float and matches the required precision
    submission_df['price'] = submission_df['price'].round(8)

    submission_file_path = 'test_out.csv'
    submission_df.to_csv(submission_file_path, index=False)

    print(f"\n--- FINAL SUBMISSION GENERATED ---")
    print(f"File: {submission_file_path} (Ready for upload)")
    print("First 5 predictions:")
    print(submission_df.head())

if __name__ == '__main__':
    try:
        train_and_predict()
    except FileNotFoundError as e:
        print(f"Error: {e}. Please ensure 'sample_test.csv' (mocking both train/test) is available.")



Processing train.csv with 100 samples...
-> Extracting Structured Features (IPQ/Unit)...
-> Mocking Text Embedding Generation (768D)...


/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

-> Mocking Image Feature Generation (512D)...


/tmp/ipython-input-2700719907.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['link_hash'] = df['image_link'].apply(lambda x: int(hashlib.sha256(str(x).encode('utf-8')).hexdigest(), 16) % (2**32))
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[IMAGE_FEATURE_COLS] = feature_matrix
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all c


Processing test.csv with 100 samples...
-> Extracting Structured Features (IPQ/Unit)...
-> Mocking Text Embedding Generation (768D)...


/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[TEXT_EMBEDDING_COLS] = embedding_matrix
/tmp/ipython-input-2700719907.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame

-> Mocking Image Feature Generation (512D)...


/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[IMAGE_FEATURE_COLS] = feature_matrix
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[IMAGE_FEATURE_COLS] = feature_matrix
/tmp/ipython-input-2700719907.py:86: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use 


--- Starting 5-Fold Multimodal LightGBM Training ---
Fold 1 Validation RMSE (Actual Price): $37.14
Fold 2 Validation RMSE (Actual Price): $22.43
Fold 3 Validation RMSE (Actual Price): $33.46
Fold 4 Validation RMSE (Actual Price): $17.88
Fold 5 Validation RMSE (Actual Price): $35.40

Average Actual Price RMSE across 5 folds: $29.26

--- FINAL SUBMISSION GENERATED ---
File: test_out.csv (Ready for upload)
First 5 predictions:
   sample_id      price
0     217392  20.290081
1     209156  17.267707
2     262333  21.279687
3     295979  24.530902
4      50604  15.932095
